# Google India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.google.com/jobs/results/?location=India

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills


Imports loaded. Date: 2026-03-27 17:10:44


In [3]:
COMPANY = "Google"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Google/Outputs/2026_03_27


In [4]:
print("=" * 60)
print("GOOGLE INDIA JOB SCRAPER")
print("Source: www.google.com/about/careers/applications/jobs/results")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


google_jobs = []
driver = setup_selenium()

try:
    url = "https://www.google.com/about/careers/applications/jobs/results/?location=India"
    driver.get(url)
    time.sleep(10)

    # Wait for results to render (Google uses heavy JS)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[class*='lLd3Je'], li[class*='sMn82b'], [data-id]"))
        )
    except:
        print("  Waiting longer for Google careers to load...")
        time.sleep(10)

    # Scroll to load more jobs (infinite scroll)
    prev_count = 0
    for scroll in range(30):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "lxml")

        # Google uses specific class names for job cards
        cards = soup.select("li[class*='lLd3Je'], li[class*='sMn82b']")
        if not cards:
            cards = soup.select("[data-id], li.result, [class*='job-result']")

        if len(cards) == prev_count and scroll > 3:
            break  # No new results loaded
        prev_count = len(cards)

        if scroll % 5 == 0:
            print(f"  Scroll {scroll+1}: {len(cards)} jobs found so far")

    # Now parse all visible cards
    soup = BeautifulSoup(driver.page_source, "lxml")
    cards = soup.select("li[class*='lLd3Je'], li[class*='sMn82b'], [data-id]")

    for card in cards:
        title_el = card.select_one("h3, h2, [class*='QJPWVe'], [class*='job-title']")
        title = title_el.get_text(strip=True) if title_el else ""

        loc_el = card.select_one("[class*='r0wTof'], [class*='location'], [class*='city']")
        loc = loc_el.get_text(strip=True) if loc_el else "India"

        link = card.select_one("a[href]")
        href = link.get("href", "") if link else ""
        job_id = card.get("data-id", href.split("/")[-1] if href else str(len(google_jobs)))

        if title and title not in [j["title"] for j in google_jobs]:
            google_jobs.append({
                "job_id": str(job_id),
                "title": title,
                "company_name": "Google",
                "raw_jd_text": card.get_text(" ", strip=True),
                "location_city": loc.split(",")[0].strip(),
                "industry": "Technology",
                "date_posted": datetime.now().strftime("%Y-%m-%d"),
                "is_active": True,
                "job_url": href if href.startswith("http") else f"https://www.google.com{href}" if href else "",
                "business_unit": "",
                "source_platform": "Google Careers",
            })

    print(f"  Found {len(google_jobs)} total Google India jobs")

    # Fetch JD for first N jobs
    if google_jobs:
        print(f"  Fetching JD details for up to 30 jobs...")
        for i, job in enumerate(google_jobs[:30]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 200:
                continue
            jd_url = f"https://www.google.com/about/careers/applications/jobs/results/{job['job_id']}"
            if href and href.startswith("/"):
                jd_url = f"https://www.google.com{href}"
            jd = fetch_jd_selenium(driver, jd_url)
            if jd:
                google_jobs[i]["raw_jd_text"] = jd
            if (i + 1) % 10 == 0:
                print(f"    Fetched {i+1}/{min(30, len(google_jobs))} JDs")

except Exception as e:
    print(f"  Error: {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"Total Google India jobs: {len(google_jobs)}")


GOOGLE INDIA JOB SCRAPER
Source: www.google.com/about/careers/applications/jobs/results


  Scroll 1: 20 jobs found so far


  Found 20 total Google India jobs
  Fetching JD details for up to 30 jobs...
Total Google India jobs: 20


In [5]:
df_google = save_results(google_jobs, "Google", OUTPUT_DIR)
if df_google is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_google.columns]
    print(df_google[cols].head(10).to_string())


  [OK] Saved 20 jobs -> Google_jobs_2026-03-27.csv
       Seniority: {'lead': 8, 'senior': 6, 'junior': 5, 'mid': 1}
       Work mode: {'onsite': 20}
       Has JD text: 20/20
       Has job URL: 20/20
       Has business unit: 0/20

Sample jobs:
                                                      title location_city seniority_level business_unit                                                                                                                      job_url
0  Technical Program Manager, Cloud, Supply Chain Analytics     Bengaluru            lead                https://www.google.comjobs/results/116843210315047622-technical-program-manager-cloud-supply-chain-analytics?location=India
1           Data Center Facilities Technician I, Electrical        Mumbai          junior                         https://www.google.comjobs/results/72640550976529094-data-center-facilities-technician-i-electrical?location=India
2                            Director, Head of Legal, India      G